# DataFrame Transformations: Select, Filter, WithColumn & Expressions

**Module**: 03_dataframe_and_spark_sql  
**Notebook**: 03_dataframe_transformations.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Complete practical guide to DataFrame transformations: select, selectExpr, filter/where, withColumn, withColumnRenamed, drop, distinct, and column functions.
- **Why does this topic exist?**: Data engineers spend 80% of their coding time transforming columns, cleaning text, and manipulating dates and timestamps.
- **What problem does it solve?**: Suboptimal expressions (like chained withColumn in tight loops) degrade Catalyst optimizer performance.
- **Where is it used in real-world Data Engineering?**: Data wrangling, feature engineering, and data cleaning pipelines.

---

## 2. Core Transformation Categories

```
+---------------------------------------------------------------+
| DATAFRAME TRANSFORMATIONS                                     |
+---------------------------------------------------------------+
| Structural:                                                   |
|  - select(), selectExpr(), withColumn(), withColumnRenamed()  |
|  - drop(), dropDuplicates()                                   |
| Row-Level Filtering:                                          |
|  - filter(), where()                                          |
| Column-Level Expressions:                                     |
|  - col(), lit(), when().otherwise()                           |
|  - String: lower, upper, trim, regexp_replace, split          |
|  - Date: to_date, to_timestamp, datediff, date_add, year      |
|  - Numeric: round, ceil, floor, abs                           |
+---------------------------------------------------------------+
```

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Transformations_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

## 3. Practical: End-to-End Data Transformation Pipeline

We transform `/opt/spark-data/retail/orders.csv` with realistic business logic.

In [ ]:
orders = spark.read.option("header", True).option("inferSchema", True).csv("/opt/spark-data/retail/orders.csv")

# Chained transformation pipeline
transformed = (
    orders
    .filter(F.col("status").isin("Completed", "Shipped"))
    .withColumn("order_date", F.to_date(F.col("order_ts")))
    .withColumn("order_year", F.year(F.col("order_date")))
    .withColumn("order_month", F.month(F.col("order_date")))
    .withColumn("gross_amount", F.round(F.col("quantity") * F.col("unit_price"), 2))
    .withColumn("discount_amount", F.round(F.col("gross_amount") * F.col("discount"), 2))
    .withColumn("net_amount", F.round(F.col("gross_amount") - F.col("discount_amount"), 2))
    .withColumn("channel_category", F.when(F.col("channel") == "Store", "Physical").otherwise("Digital"))
    .select(
        "order_id", "customer_id", "order_date", "order_year", "order_month",
        "gross_amount", "discount_amount", "net_amount", "channel_category", "status"
    )
)

transformed.show(5)

## 4. High-Performance Column Replacement with `select` vs Multiple `withColumn`

A common anti-pattern is calling `.withColumn()` 50 times in a row, which bloats the Catalyst tree. Using `select` or `selectExpr` generates a cleaner plan.

In [ ]:
# High performance multi-column projection
clean_df = orders.select(
    F.col("order_id"),
    F.col("customer_id"),
    F.col("quantity"),
    F.col("unit_price"),
    (F.col("quantity") * F.col("unit_price")).alias("raw_total"),
    F.upper(F.col("channel")).alias("channel_upper")
)
clean_df.show(3)

---

## Experiment: Chained withColumn vs Projection Plan Complexity

- **1. What are we testing?**: Compare the analyzed logical plan tree depth between 10 chained withColumn calls vs a single select projection.
- **2. What do we expect?**: 10 withColumns creates 10 nested Project operators; a single select creates exactly 1 Project operator.

In [ ]:
# 10 chained withColumns
df_chained = orders
for i in range(10):
    df_chained = df_chained.withColumn(f"col_{i}", F.lit(i))

# Single select
df_single = orders.select("*", *[F.lit(i).alias(f"col_{i}") for i in range(10)])

print(f"Chained withColumn Plan Tree Depth: {len(df_chained._jdf.queryExecution().analyzed().treeString().splitlines())}")
print(f"Single Select Plan Tree Depth:      {len(df_single._jdf.queryExecution().analyzed().treeString().splitlines())}")

**3. What happened?**: Chained withColumns generated a massive nested logical plan tree.  
**4. Why did it happen?**: Each withColumn creates a new DataFrame wrapper node in the unresolved logical tree.  
**5. Practical Engineering Lesson**: When adding or transforming multiple columns, prefer a single select() or selectExpr() call.

---

## Practical Exercises

### Exercise 1 (Beginner): Rename and Cast Columns
**Problem Statement**: Rename'quantity'to'units_ordered'and cast'unit_price'to DecimalType(10, 2).  
**Input / Context**: orders DataFrame

In [ ]:
# Exercise 1: Rename and cast

### Exercise 2 (Intermediate): Extract Domain from Customer Email
**Problem Statement**: Load customers.csv and use regexp_extract or split to extract email domains into a'domain'column.  
**Input / Context**: /opt/spark-data/retail/customers.csv

In [ ]:
# Exercise 2: Email domain extraction

### Exercise 3 (Advanced): Conditional Multi-Tier Discount Calculation
**Problem Statement**: Calculate a loyalty discount percentage: Gold = 20%, Silver = 10%, Bronze/Consumer = 0% using nested when().otherwise().  
**Input / Context**: custs DataFrame

In [ ]:
# Exercise 3: Loyalty discount tier

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Rename and Cast Columns):
```python
ex1 = orders.withColumnRenamed("quantity","units_ordered").withColumn("unit_price", F.col("unit_price").cast("decimal(10,2)"))
ex1.printSchema()
```

#### Solution 2 (Extract Domain from Customer Email):
```python
custs = spark.read.option("header", True).csv("/opt/spark-data/retail/customers.csv")
custs_domain = custs.withColumn("domain", F.split(F.col("email"),"@").getItem(1))
custs_domain.select("email","domain").show(5)
```

#### Solution 3 (Conditional Multi-Tier Discount Calculation):
```python
discount_df = custs.withColumn("tier_discount",
    F.when(F.col("loyalty_tier") =="Gold", 0.20)
     .when(F.col("loyalty_tier") =="Silver", 0.10)
     .otherwise(0.0)
)
discount_df.select("first_name","loyalty_tier","tier_discount").show(5)
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Vectorized Transformations**: Always use built-in pyspark.sql.functions for string, date, and math operations.
- **Single Projection**: Use select() with expressions rather than dozens of chained withColumns.
- **Column Expressions**: Columns can be dynamically created with F.col() and F.lit().

### Essential Technical Terms
- `selectExpr`: Select projection accepting raw SQL string expressions.
- `when().otherwise()`: Conditional case-when statement.
- `cast()`: Converts a column to another DataType.

### Common Engineering Mistakes
- **50 withColumn calls in a loop**: Causes excessive plan complexity and slow analysis.
- **Missing lit() for constants**: Passing raw Python values into col() throws type errors.

### Interview Questions on This Topic
- **Q: What is the difference between filter() and where()?**
  - *A*: They are exact aliases of each other.
- **Q: Why is select() preferred over multiple withColumns?**
  - *A*: Creates a single Project operator in the logical plan rather than nesting dozens of intermediate projections.

### What to Study Next
Open 03_dataframe_and_spark_sql/04_spark_sql.ipynb to master SQL queries and temporary views.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')